In [ ]:
# %% [markdown]
# # Log Exploration Template
# Einfach LOG_PATH anpassen und alle Zellen nacheinander ausführen.

# %%
import pandas as pd
import pm4py
import matplotlib.pyplot as plt

LOG_PATH = "pfad/zum/log.xes"  # hier anpassen
CASE_COL = "case:concept:name"
ACT_COL  = "concept:name"
TS_COL   = "time:timestamp"
RES_COL  = "org:resource"

df = pm4py.read_xes(LOG_PATH)
df = df.sort_values([CASE_COL, TS_COL]).reset_index(drop=True)
df[TS_COL] = pd.to_datetime(df[TS_COL], utc=True).dt.tz_localize(None)

# %% [markdown]
# ## 1. Grundinfos

# %%
print(f"Events:     {len(df):,}")
print(f"Cases:      {df[CASE_COL].nunique():,}")
print(f"Activities: {df[ACT_COL].nunique()}")
print(f"Zeitraum:   {df[TS_COL].min().date()} → {df[TS_COL].max().date()}")
print(f"Spalten:    {list(df.columns)}")

# %% [markdown]
# ## 2. Activity-Verteilung

# %%
print(df[ACT_COL].value_counts())

# %% [markdown]
# ## 3. Ressourcen-Analyse

# %%
if RES_COL in df.columns:
    print(f"Ressourcen gesamt: {df[RES_COL].nunique()}")
    print(f"Fehlende Werte:    {df[RES_COL].isna().sum()}")
    print()
    print(df[RES_COL].value_counts().head(20))
else:
    print("Keine Ressourcen-Spalte vorhanden!")

# %% [markdown]
# ## 4. Ressourcen – auffällige Muster (wie User_1 bei BPI 2017)

# %%
if RES_COL in df.columns:
    for res, group in df.groupby(RES_COL):
        act_dist     = group[ACT_COL].value_counts(normalize=True)
        events_p_case = group.groupby(CASE_COL).size()
        hour_dist    = group[TS_COL].dt.hour.value_counts().sort_index()
        
        # auffällig wenn: eine Activity >90%, fast immer 1 Event pro Case, enge Stundenfenster
        dominant_act   = act_dist.iloc[0] > 0.90
        single_event   = (events_p_case == 1).mean() > 0.90
        narrow_hours   = hour_dist[hour_dist > 0].index.nunique() <= 3
        
        if dominant_act and single_event:
            print(f"⚠️  Auffällig: {res}")
            print(f"   Häufigste Activity: {act_dist.index[0]} ({act_dist.iloc[0]:.0%})")
            print(f"   Events pro Case:    {events_p_case.mean():.2f} (median {events_p_case.median():.0f})")
            print(f"   Aktive Stunden:     {sorted(hour_dist[hour_dist > 0].index.tolist())}")
            print()

# %% [markdown]
# ## 5. Case Duration

# %%
case_duration = (
    df.groupby(CASE_COL)[TS_COL]
      .agg(start="min", end="max")
)
duration_h = (case_duration["end"] - case_duration["start"]).dt.total_seconds() / 3600

print(f"min:    {duration_h.min():.1f} h")
print(f"mean:   {duration_h.mean():.1f} h")
print(f"median: {duration_h.median():.1f} h")
print(f"max:    {duration_h.max():.1f} h")

duration_h.hist(bins=50)
plt.xlabel("Case Duration (h)")
plt.ylabel("Anzahl Cases")
plt.title("Case Duration Verteilung")
plt.show()

# %% [markdown]
# ## 6. Events pro Case

# %%
events_per_case = df.groupby(CASE_COL).size()

print(f"min:    {events_per_case.min()}")
print(f"mean:   {events_per_case.mean():.1f}")
print(f"median: {events_per_case.median():.0f}")
print(f"max:    {events_per_case.max()}")

events_per_case.hist(bins=50)
plt.xlabel("Events pro Case")
plt.ylabel("Anzahl Cases")
plt.title("Events pro Case Verteilung")
plt.show()

# %% [markdown]
# ## 7. Fehlende Werte

# %%
print(df.isnull().sum()[df.isnull().sum() > 0])

# %% [markdown]
# ## 8. Batch-Anteil checken (±5 Min Fenster)

# %%
import numpy as np

df_sorted = df.sort_values(TS_COL)
batch_counts = []

for act, group in df_sorted.groupby(ACT_COL):
    timestamps = group[TS_COL].values.astype("int64") // 1_000_000_000
    for ts in timestamps:
        window = np.sum((timestamps >= ts - 300) & (timestamps <= ts + 300)) - 1
        batch_counts.append(window > 0)

print(f"Batch-Anteil: {np.mean(batch_counts):.1%} der Events")